# Steamroller — a quantitative teardown 🔬
### The carry premium by rate bucket · Newey-West t · the negative-skew crash · vol-managed carry · the UIRP null

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Crash risk?: Severe](https://img.shields.io/badge/Crash_risk%3F-Severe-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §8.2, the FX carry trade: UIRP fails, so high-rate currencies earn a premium. We prove the engine on a synthetic G10 with a baked premium and risk-off crashes, then read the real verdict off G10 FRED data (`examples/verify.py --fetch`).

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real G10 run is in [`../docs/results.md`](../docs/results.md) (needs one FRED fetch), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from steamroller import data, carry, strategy, decompose, extension

# Offline synthetic G10: a CARRY PREMIUM tape (high-rate currencies out-earn, punctuated by sticky
# risk-off crashes) and a full-UIRP NULL. The real G10 verdict (FRED) is in ../docs/results.md.
xr,  rates,  truth = data.synthetic_carry(carry_strength=0.9, seed=27)   # the carry-premium tape
xr0, rates0, _     = data.synthetic_carry(carry_strength=0.0, seed=27)   # the full-UIRP null
print(f"{truth.n_ccy} currencies x {truth.n_months} months | baked carry_strength={truth.carry_strength} | null=0")


9 currencies x 600 months | baked carry_strength=0.9 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — is the premium real? | 🟢 `REAL` | High-rate currencies out-earn; synthetic control premium **+2.1%/yr** (HAC *t* **+2.8**), null flat (*t* **-0.2**); decades of academic evidence (Lustig-Verdelhan; Menkhoff et al.). |
| **Tradability** | 🟡 `FRAGILE` | Low turnover (survives costs), but a fat negative tail: skew **-1.54**, drawdown **-28%**. |
| **Crash risk?** | ⚪ `Severe` | The crash is a sudden risk-off jump that vol-targeting can't forecast — vol-managing lifts the Sharpe but deepens the drawdown ( → **-44%**). |

> **In one sentence:** the carry premium is real, durable and cheap to run — and it is compensation for a sharply negative-skewed crash that the desk's usual vol overlay cannot dodge.

*(This notebook executes on the synthetic control; the real G10 numbers come from [`../docs/results.md`](../docs/results.md) after `examples/verify.py --fetch`.)*

## Beat 1 · The claim, precisely

UIRP: $(1+r_d) = \mathbb{E}_t[S_{t+T}]/S_t \cdot (1+r_f)$ — the high-rate currency should depreciate to offset its yield. Empirically it doesn't, so the excess return of holding currency $i$ funded in the base is $x_i \approx (r_i - r_{\text{base}}) + \Delta\log S_i > 0$ for high $r_i$. The carry portfolio is long the top-rate, short the bottom-rate tercile. The synthetic bakes partial UIRP (the premium) plus sticky risk-off crashes; `carry_strength = 0` is full UIRP (the null).

In [2]:
pb = carry.carry_premium_by_bucket(xr, rates)
print(f"high-minus-low rate bucket spread {pb['hml_ann_pct']:+.1f}%/yr (synthetic); "
      f"null {carry.carry_premium_by_bucket(xr0, rates0)['hml_ann_pct']:+.1f}%/yr")

high-minus-low rate bucket spread +4.2%/yr (synthetic); null -0.1%/yr


## Beat 2 · So what?

Carry's defining feature is its **skewness**, not its mean. Brunnermeier, Nagel & Pedersen (2008) tie carry crashes to the sudden unwinding of crowded, leveraged positions when funding liquidity dries up — a jump, correlated across all carry pairs at once. So the premium is a risk premium for a crash, and the open questions are: is it significant, how fat is the tail, and does any cheap overlay shrink it. Beats 4–6 answer all three.

## Beat 3 · Pre-registered protocol

1. **Premium** (`carry.carry_premium_by_bucket`, `decompose.premium_tstat`): bucket spread + HAC *t*. Real ⇔ positive, |t|>2.
2. **Crash** (`decompose.crash_profile`, `downside_concentration`): skew, worst months, drawdown.
3. **Risk management** (`extension.crash_comparison`): vol-managed vs plain.
4. **Null:** full UIRP collapses the premium.

**Verdict logic:** `REAL` premium, `FRAGILE`/`Severe` if the tail is deep and unhedgeable.

## Beat 4 · The teardown

### 4a · Premium and t, carry vs null

In [3]:
for label, (x, rt) in [('carry', (xr, rates)), ('null', (xr0, rates0))]:
    pt = decompose.premium_tstat(x, rt, cost_bps=10.0); cmp = strategy.compare(x, rt, cost_bps=10.0)
    print(f"{label:6s}: premium {pt['mean_ann_pct']:+.1f}%/yr (HAC t {pt['t_stat']:+.1f}), "
          f"Sharpe {cmp['sharpe']:+.2f}, turnover {cmp['turnover_ann']:.1f}x")

carry : premium +2.1%/yr (HAC t +2.8), Sharpe +0.60, turnover 0.0x


null  : premium -0.1%/yr (HAC t -0.2), Sharpe -0.03, turnover 0.0x


### 4b · The crash profile and downside concentration

In [4]:
cr = decompose.crash_profile(xr, rates, cost_bps=10.0)
dc = decompose.downside_concentration(xr, rates, cost_bps=10.0, k=5)
print(f"skew {cr['skew']:+.2f}, worst month {cr['worst_month_pct']:+.1f}%, worst-5 {cr['worst5_months_mean_pct']:+.1f}%, max drawdown {cr['max_drawdown_pct']:.0f}%")
print(f"the worst 5 months carry {dc['worst_k_share_of_losses']:.0%} of all losing-month losses -- crash-concentrated")

skew -1.54, worst month -4.3%, worst-5 -3.8%, max drawdown -28%
the worst 5 months carry 11% of all losing-month losses -- crash-concentrated


> 💡 **In plain words.** Carry doesn't lose a little often; it loses a lot, rarely, all at once. The mean looks like a smooth yield, but a handful of months hold most of the pain — the signature of a risk premium for a tail event.

### 4c · Vol-management lifts Sharpe, not the tail

In [5]:
cc = extension.crash_comparison(xr, rates, cost_bps=10.0)
import pandas as pd; display(pd.DataFrame(cc).T.round(2))
print('Sharpe up, drawdown not down: the crash is a jump trailing vol cannot forecast.')

,sharpe,skew,worst_month_pct,max_drawdown_pct
plain,0.6800,-1.5700,-4.3200,-28.2700
managed,0.9500,-1.0100,-10.3600,-43.5300


Sharpe up, drawdown not down: the crash is a jump trailing vol cannot forecast.


## Beat 5 · The verdict

- **Real premium** (4a): +2.1%/yr (*t* +2.8); null flat.
- **Severe, concentrated tail** (4b): skew -1.54, drawdown -28%.
- **Unhedgeable by vol-targeting** (4c): Sharpe up, drawdown not.

> **Signal `REAL` · Tradability `FRAGILE` · Crash risk? `Severe`.**

## Beat 6 · Could you trade it?

- **Cheap to run** (slow rates ⇒ low turnover) — it survives costs.
- **But the drawdown is a career risk**, correlated across all carry pairs.
- **Vol-targeting fails** — it can lever you *into* the jump.

Tradability **`FRAGILE`**; crash **`Severe`**.

## Beat 7 · Going further

### 7a · Worked complement — why risk management doesn't dodge the steamroller
Plain vs vol-managed carry, side by side; the Sharpe rises but the drawdown doesn't fall.

In [6]:
cc = extension.crash_comparison(xr, rates, cost_bps=10.0)
for k in ['plain', 'managed']:
    p = cc[k]; print(f"{k:8s}: Sharpe {p['sharpe']:+.2f}, skew {p['skew']:+.2f}, worst month {p['worst_month_pct']:+.1f}%, max drawdown {p['max_drawdown_pct']:.0f}%")
print('Real G10 (../docs/results.md after --fetch): same shape -- Sharpe up, drawdown -28%-class unchanged.')

plain   : Sharpe +0.68, skew -1.57, worst month -4.3%, max drawdown -28%
managed : Sharpe +0.95, skew -1.01, worst month -10.4%, max drawdown -44%
Real G10 (../docs/results.md after --fetch): same shape -- Sharpe up, drawdown -28%-class unchanged.


**The result.** Vol-targeting — the overlay that earned [Study 16](../../16-storm-shy/) the desk's only green and tamed momentum's crash in [Study 24](../../24-stampede/) — *fails* on carry. It lifts the standalone Sharpe (it cuts exposure in noisy calm spells) but does not shrink the drawdown, and can deepen it by levering up into a calm that precedes a jump. That is the cleanest statement of *why* carry's crash is `Severe`: it is the one tail on this desk that a trailing-volatility forecast cannot see. The honest defences — options, a risk-off switch, cross-asset diversification — are forks, not a vol target. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **The real G10 tape** (`verify.py --fetch`) — the actual 1998/2008 crashes, fingerprinted.
- **Risk-off conditioning** — gate the book on a VIX / drawdown / funding-stress signal (Brunnermeier-Nagel-Pedersen) and see if it dodges the jump without killing the premium.
- **Options tail hedge** — price the put protection against the carry premium; is the net still positive?

PRs welcome — run the real tape, or build a tail hedge that earns its keep.